In [ ]:
import os
print(os.getcwd())

/content


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving IPL_Ball_by_Ball_2008_2022.csv to IPL_Ball_by_Ball_2008_2022.csv


In [ ]:
import pandas as pd

df = pd.read_csv("IPL_Ball_by_Ball_2008_2022.csv")
print(df.head())

        ID  innings  overs  ballnumber       batter          bowler  \
0  1312200        1      0           1  YBK Jaiswal  Mohammed Shami   
1  1312200        1      0           2  YBK Jaiswal  Mohammed Shami   
2  1312200        1      0           3   JC Buttler  Mohammed Shami   
3  1312200        1      0           4  YBK Jaiswal  Mohammed Shami   
4  1312200        1      0           5  YBK Jaiswal  Mohammed Shami   

   non-striker extra_type  batsman_run  extras_run  total_run  non_boundary  \
0   JC Buttler        NaN            0           0          0             0   
1   JC Buttler    legbyes            0           1          1             0   
2  YBK Jaiswal        NaN            1           0          1             0   
3   JC Buttler        NaN            0           0          0             0   
4   JC Buttler        NaN            0           0          0             0   

   isWicketDelivery player_out kind fielders_involved       BattingTeam  
0                 0     

Q1.Load the dataset.Create a Series of total runs per batsman with player names as the index. Examine run distribution using descriptive statistics and identify the top 10 run-scorers.

In [ ]:
# Check columns
print(df.columns)


Index(['ID', 'innings', 'overs', 'ballnumber', 'batter', 'bowler',
       'non-striker', 'extra_type', 'batsman_run', 'extras_run', 'total_run',
       'non_boundary', 'isWicketDelivery', 'player_out', 'kind',
       'fielders_involved', 'BattingTeam'],
      dtype='object')


In [ ]:
# Create Series
runs_per_batter = df.groupby('batter')['batsman_run'].sum()

# Display first few
print("Runs per batter:\n", runs_per_batter.head())

# Descriptive statistics
print("\nDescriptive Statistics:\n", runs_per_batter.describe())

# Top 10 scorers
top10 = runs_per_batter.sort_values(ascending=False).head(10)

print("\nTop 10 Run Scorers:\n", top10)

Runs per batter:
 batter
A Ashish Reddy    280
A Badoni          161
A Chandila          4
A Chopra           53
A Choudhary        25
Name: batsman_run, dtype: int64

Descriptive Statistics:
 count     605.000000
mean      464.428099
std       985.272855
min         0.000000
25%        15.000000
50%        73.000000
75%       326.000000
max      6634.000000
Name: batsman_run, dtype: float64

Top 10 Run Scorers:
 batter
V Kohli           6634
S Dhawan          6244
DA Warner         5883
RG Sharma         5881
SK Raina          5536
AB de Villiers    5181
CH Gayle          4997
MS Dhoni          4978
RV Uthappa        4954
KD Karthik        4377
Name: batsman_run, dtype: int64


Q2. Compute the strike rate (runs / balls faced × 100)
of top batsmen using ufuncs. Compare performance in
the powerplay (overs 1–6) versus death overs (overs
17–20).


In [ ]:
import pandas as pd
import numpy as np

# Valid balls (exclude wides)
df['valid_ball'] = np.where(df['extra_type'] == 'wides', 0, 1)

# Runs and balls
runs = df.groupby('batter')['batsman_run'].sum()
balls = df.groupby('batter')['valid_ball'].sum()

# Strike rate
strike_rate = np.divide(runs, balls) * 100

# Top batsmen filter
top_batters = runs[runs > 500].index

# Top 10 strike rates
top10_sr = strike_rate[top_batters].sort_values(ascending=False).head(10)

print("Top 10 Strike Rates:\n", top10_sr)

# Powerplay
powerplay = df[(df['overs'] >= 1) & (df['overs'] <= 6)]
runs_pp = powerplay.groupby('batter')['batsman_run'].sum()
balls_pp = powerplay.groupby('batter')['valid_ball'].sum()
sr_pp = np.divide(runs_pp, balls_pp) * 100

# Death overs
death = df[(df['overs'] >= 17) & (df['overs'] <= 20)]
runs_death = death.groupby('batter')['batsman_run'].sum()
balls_death = death.groupby('batter')['valid_ball'].sum()
sr_death = np.divide(runs_death, balls_death) * 100

# Comparison
# Comparison DataFrame
comparison = pd.DataFrame({
    'Powerplay_SR': sr_pp,
    'Death_SR': sr_death
}).dropna()

# Fix: take only common players
common_players = comparison.index.intersection(top_batters)

comparison_top = comparison.loc[common_players]

print("\nComparison:\n",
      comparison_top.sort_values(by='Death_SR', ascending=False).head(10))

Top 10 Strike Rates:
 batter
AD Russell        177.768091
LS Livingstone    166.869301
SP Narine         162.698413
V Sehwag          155.441595
CH Morris         155.276382
GJ Maxwell        153.846154
SO Hetmyer        152.197802
AB de Villiers    151.890941
N Pooran          150.743802
JC Buttler        149.603803
dtype: float64

Comparison:
                 Powerplay_SR    Death_SR
batter                                  
AB de Villiers    116.944801  251.824818
Q de Kock         138.558559  247.058824
V Sehwag          149.594320  242.857143
MA Agarwal        126.511628  241.818182
CJ Anderson        81.250000  232.000000
CH Gayle          144.354336  228.571429
Ishan Kishan      123.022847  224.528302
SV Samson         124.299065  220.833333
AD Russell        226.086957  219.776119
V Kohli           112.453234  216.346154


Q3. Apply hierarchical indexing on 'batting_team' and
'over'. Examine average runs per over at each
hierarchical level using xs() and groupby. Identify team scoring patterns.


In [ ]:

# OPTIONAL: convert overs to actual (1–20 instead of 0–19)
df['over_actual'] = df['overs'] + 1

#  Create MultiIndex
df_multi = df.set_index(['BattingTeam', 'over_actual'])

#  Step 1: Runs per match per over (important for correct average)
runs_per_over = df_multi.groupby(['BattingTeam', 'ID', 'over_actual'])['total_run'].sum()

#  Step 2: Average runs per over per team
avg_runs = runs_per_over.groupby(['BattingTeam', 'over_actual']).mean()

print("Average Runs per Over:\n", avg_runs.head())

#  xs() example – specific team
print("\nMumbai Indians Data:\n", avg_runs.xs('Mumbai Indians', level='BattingTeam'))

#  xs() example – Over 20 (now correct)
print("\nOver 20 Data:\n", avg_runs.xs(20, level='over_actual'))

#  Pivot table for analysis
pivot_table = avg_runs.unstack(level='over_actual')

print("\nPivot Table:\n", pivot_table)

Average Runs per Over:
 BattingTeam          over_actual
Chennai Super Kings  1              5.240385
                     2              6.504808
                     3              7.629808
                     4              8.326923
                     5              8.576923
Name: total_run, dtype: float64

Mumbai Indians Data:
 over_actual
1      5.956710
2      6.688312
3      7.822511
4      8.406926
5      8.380952
6      8.411255
7      6.426087
8      7.078261
9      7.626087
10     7.148472
11     7.301310
12     7.882096
13     8.048035
14     8.273128
15     8.555556
16     8.995475
17     9.481651
18    10.358491
19    10.296651
20    11.241935
Name: total_run, dtype: float64

Over 20 Data:
 BattingTeam
Chennai Super Kings            11.578947
Deccan Chargers                 8.825397
Delhi Capitals                 10.638298
Delhi Daredevils               10.548673
Gujarat Lions                   8.761905
Gujarat Titans                 11.866667
Kings XI Punjab          

Q4. Create a &#39;Player Impact Score&#39; combining runs,
fours, sixes, and wickets using normalized ufunc
weights. Rank all players and construct a season-wise
impact leaderboard.

In [ ]:
# -------------------------------
# Step 2: Overall Player Metrics
# -------------------------------
runs = df.groupby('batter')['batsman_run'].sum()
fours = df[df['batsman_run'] == 4].groupby('batter').size()
sixes = df[df['batsman_run'] == 6].groupby('batter').size()
wickets = df[df['isWicketDelivery'] == 1].groupby('bowler').size()

# Combine all metrics
impact_df = pd.DataFrame({
    'runs': runs,
    'fours': fours,
    'sixes': sixes,
    'wickets': wickets
}).fillna(0)

# -------------------------------
# Step 3: Normalize Data
# -------------------------------
impact_norm = (impact_df - impact_df.min()) / (impact_df.max() - impact_df.min())

# -------------------------------
# Step 4: Assign Weights
# -------------------------------
weights = {
    'runs': 0.4,
    'fours': 0.2,
    'sixes': 0.2,
    'wickets': 0.2
}

# -------------------------------
# Step 5: Compute Impact Score (ufunc)
# -------------------------------
impact_df['Impact_Score'] = (
    np.multiply(impact_norm['runs'], weights['runs']) +
    np.multiply(impact_norm['fours'], weights['fours']) +
    np.multiply(impact_norm['sixes'], weights['sixes']) +
    np.multiply(impact_norm['wickets'], weights['wickets'])
)

# -------------------------------
# Step 6: Rank Players (Overall)
# -------------------------------
ranking = impact_df.sort_values(by='Impact_Score', ascending=False)

print("Top 10 Players Overall:\n")
print(ranking.head(10))


# =====================================================
# 🔥 SEASON-WISE IMPACT LEADERBOARD
# =====================================================

# -------------------------------
# Step 7: Create Season Column
# -------------------------------
# If dataset already has 'season', skip this
df['season'] = df['ID'] // 1000000   # approximation

# -------------------------------
# Step 8: Season-wise Metrics
# -------------------------------
season_runs = df.groupby(['season', 'batter'])['batsman_run'].sum()
season_fours = df[df['batsman_run'] == 4].groupby(['season', 'batter']).size()
season_sixes = df[df['batsman_run'] == 6].groupby(['season', 'batter']).size()
season_wickets = df[df['isWicketDelivery'] == 1].groupby(['season', 'bowler']).size()

# Combine season metrics
season_df = pd.DataFrame({
    'runs': season_runs,
    'fours': season_fours,
    'sixes': season_sixes,
    'wickets': season_wickets
}).fillna(0)

# -------------------------------
# Step 9: Normalize Season Data
# -------------------------------
season_norm = (season_df - season_df.min()) / (season_df.max() - season_df.min())

# -------------------------------
# Step 10: Compute Season Impact Score
# -------------------------------
season_df['Impact_Score'] = (
    np.multiply(season_norm['runs'], weights['runs']) +
    np.multiply(season_norm['fours'], weights['fours']) +
    np.multiply(season_norm['sixes'], weights['sixes']) +
    np.multiply(season_norm['wickets'], weights['wickets'])
)

# -------------------------------
# Step 11: Rank Players per Season
# -------------------------------
season_df['Rank'] = season_df.groupby('season')['Impact_Score']\
                            .rank(ascending=False)

leaderboard = season_df[season_df['Rank'] <= 10]

print("\nSeason-wise Top 10 Players:\n")
print(leaderboard)

Top 10 Players Overall:

                  runs  fours  sixes  wickets  Impact_Score
V Kohli         6634.0  581.0  219.0      5.0      0.692600
S Dhawan        6244.0  701.0  137.0      4.0      0.656673
RG Sharma       5881.0  519.0  241.0     16.0      0.652392
DA Warner       5883.0  577.0  216.0      0.0      0.639674
CH Gayle        4997.0  408.0  359.0     19.0      0.636059
SK Raina        5536.0  506.0  204.0     30.0      0.620795
AB de Villiers  5181.0  414.0  253.0      0.0      0.571455
SR Watson       3880.0  377.0  190.0    107.0      0.550738
RV Uthappa      4954.0  481.0  182.0      0.0      0.537329
MS Dhoni        4978.0  346.0  229.0      0.0      0.526443

Season-wise Top 10 Players:

                         runs  fours  sixes  wickets  Impact_Score  Rank
season                                                                  
0      AB de Villiers  3270.0  275.0  142.0      0.0      0.560892   9.0
       CH Gayle        3451.0  282.0  252.0     19.0      0.693004

Q5. Examine missing values in dismissal-related
columns. Determine the effect of replacing NaN with
&#39;not out&#39; versus dropping those rows on overall bowling
statistics.

In [ ]:
# -------------------------------
# Step 2: Check Missing Values
# -------------------------------
print("Missing Values:\n")
print(df[['player_out', 'kind', 'fielders_involved']].isnull().sum())

# -------------------------------
# Step 3: Bowling Stats (Original)
# -------------------------------
# Total wickets per bowler
wickets_original = df[df['isWicketDelivery'] == 1].groupby('bowler').size()

# Balls bowled (excluding wides)
df['valid_ball'] = np.where(df['extra_type'] == 'wides', 0, 1)
balls = df.groupby('bowler')['valid_ball'].sum()

# Bowling average = runs conceded / wickets
runs_conceded = df.groupby('bowler')['total_run'].sum()

bowling_avg_original = runs_conceded / wickets_original

print("\nOriginal Bowling Average:\n", bowling_avg_original.head())


# =====================================================
# 🔹 CASE 1: Replace NaN with "not out"
# =====================================================

df_replace = df.copy()

df_replace['player_out'] = df_replace['player_out'].fillna('not out')
df_replace['kind'] = df_replace['kind'].fillna('not out')

# Recalculate wickets
wickets_replace = df_replace[df_replace['isWicketDelivery'] == 1]\
                    .groupby('bowler').size()

bowling_avg_replace = runs_conceded / wickets_replace

print("\nBowling Average (NaN → 'not out'):\n", bowling_avg_replace.head())


# =====================================================
# 🔹 CASE 2: Drop rows with NaN
# =====================================================

df_drop = df.dropna(subset=['player_out', 'kind'])

# Recalculate wickets
wickets_drop = df_drop[df_drop['isWicketDelivery'] == 1]\
                .groupby('bowler').size()

# Runs and balls also change
runs_drop = df_drop.groupby('bowler')['total_run'].sum()
balls_drop = df_drop.groupby('bowler')['valid_ball'].sum()

bowling_avg_drop = runs_drop / wickets_drop

print("\nBowling Average (After Dropping NaN):\n", bowling_avg_drop.head())


# =====================================================
# 🔹 Step 6: Comparison
# =====================================================

comparison = pd.DataFrame({
    'Original': bowling_avg_original,
    'NaN_Replaced': bowling_avg_replace,
    'NaN_Dropped': bowling_avg_drop
})

print("\nComparison of Bowling Averages:\n")
print(comparison.head(10))

Missing Values:

player_out           214803
kind                 214803
fielders_involved    217966
dtype: int64

Original Bowling Average:
 bowler
A Ashish Reddy    21.052632
A Badoni           6.000000
A Chandila        22.272727
A Choudhary       28.800000
A Dananjaya             NaN
dtype: float64

Bowling Average (NaN → 'not out'):
 bowler
A Ashish Reddy    21.052632
A Badoni           6.000000
A Chandila        22.272727
A Choudhary       28.800000
A Dananjaya             NaN
dtype: float64

Bowling Average (After Dropping NaN):
 bowler
A Ashish Reddy    0.0
A Badoni          0.0
A Chandila        0.0
A Choudhary       0.0
A Flintoff        0.0
dtype: float64

Comparison of Bowling Averages:

                 Original  NaN_Replaced  NaN_Dropped
bowler                                              
A Ashish Reddy  21.052632     21.052632     0.000000
A Badoni         6.000000      6.000000     0.000000
A Chandila      22.272727     22.272727     0.000000
A Choudhary     28.800000 

Q6. Using boolean indexing, filter deliveries where
extras are greater than zero. Examine which teams
concede the most extras and in which match phases
they occur.

In [ ]:


# -------------------------------
# Step 2: Boolean Indexing (Extras > 0)
# -------------------------------
extras_df = df[df['extras_run'] > 0]

print("Sample Extras Data:\n", extras_df.head())


# =====================================================
# 🔹 Step 3: Identify Bowling Team (Conceding Team)
# =====================================================
# Assuming two teams per match:
# If BattingTeam is one, other is bowling team
# If dataset has BowlingTeam column, use directly

if 'BowlingTeam' not in df.columns:
    # Approximation: we assume match teams alternate (simplified)
    df['BowlingTeam'] = df.groupby('ID')['BattingTeam']\
                         .transform(lambda x: x.iloc[0] if x.iloc[0] != x.iloc[-1] else x.iloc[-1])

# Now filter extras again with bowling team
extras_df = df[df['extras_run'] > 0]


# =====================================================
# 🔹 Step 4: Teams conceding most extras
# =====================================================
extras_by_team = extras_df.groupby('BowlingTeam')['extras_run'].sum()

print("\nExtras conceded by teams:\n", extras_by_team.sort_values(ascending=False).head(10))


# =====================================================
# 🔹 Step 5: Define Match Phases
# =====================================================
def phase(over):
    if over <= 5:
        return 'Powerplay'
    elif over <= 15:
        return 'Middle Overs'
    else:
        return 'Death Overs'

extras_df['Phase'] = extras_df['overs'].apply(phase)


# =====================================================
# 🔹 Step 6: Extras by Phase
# =====================================================
extras_by_phase = extras_df.groupby('Phase')['extras_run'].sum()

print("\nExtras by Phase:\n", extras_by_phase)


# =====================================================
# 🔹 Step 7: Team vs Phase Analysis
# =====================================================
team_phase = extras_df.groupby(['BowlingTeam', 'Phase'])['extras_run'].sum().unstack()

print("\nTeam vs Phase Extras:\n", team_phase)

Sample Extras Data:
           ID  innings  overs  ballnumber       batter             bowler  \
1    1312200        1      0           2  YBK Jaiswal     Mohammed Shami   
65   1312200        1     10           6   D Padikkal          HH Pandya   
138  1312200        2      3           1      MS Wade  M Prasidh Krishna   
148  1312200        2      4           4    HH Pandya           TA Boult   
151  1312200        2      4           7    HH Pandya           TA Boult   

      non-striker extra_type  batsman_run  extras_run  total_run  \
1      JC Buttler    legbyes            0           1          1   
65     JC Buttler    legbyes            0           1          1   
138  Shubman Gill      wides            0           5          5   
148  Shubman Gill      wides            0           1          1   
151  Shubman Gill      wides            0           1          1   

     non_boundary  isWicketDelivery player_out kind fielders_involved  \
1               0                 0     

/tmp/ipykernel_28744/3026286268.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  extras_df['Phase'] = extras_df['overs'].apply(phase)


Q7. Design a match-phase classifier using apply()
that labels each delivery as &#39;Powerplay&#39;, &#39;Middle&#39;, or
&#39;Death&#39; overs. Build a team-wise phase performance
matrix using pivot_table.

In [ ]:

# Step 2: Create Match Phase Classifier using apply()
# -------------------------------
def classify_phase(over):
    if over <= 5:
        return 'Powerplay'
    elif over <= 15:
        return 'Middle'
    else:
        return 'Death'

df['Phase'] = df['overs'].apply(classify_phase)

print("Sample Phase Classification:\n", df[['overs', 'Phase']].head())


# -------------------------------
# Step 3: Team-wise Phase Performance
# -------------------------------
# Using pivot_table

phase_matrix = pd.pivot_table(
    df,
    values='total_run',          # performance metric
    index='BattingTeam',         # rows = teams
    columns='Phase',             # columns = match phases
    aggfunc='sum'                # total runs
)

print("\nTeam-wise Phase Performance Matrix:\n")
print(phase_matrix)


# -------------------------------
# Step 4: Average Runs per Phase (Optional Better Analysis)
# -------------------------------
phase_avg = pd.pivot_table(
    df,
    values='total_run',
    index='BattingTeam',
    columns='Phase',
    aggfunc='mean'
)

print("\nAverage Runs per Phase:\n")
print(phase_avg)

Sample Phase Classification:
    overs      Phase
0      0  Powerplay
1      0  Powerplay
2      0  Powerplay
3      0  Powerplay
4      0  Powerplay

Team-wise Phase Performance Matrix:

Phase                        Death  Middle  Powerplay
BattingTeam                                          
Chennai Super Kings           7871   16107       9415
Deccan Chargers               2539    5507       3417
Delhi Capitals                2175    4850       3120
Delhi Daredevils              5043   11893       7360
Gujarat Lions                  921    2382       1559
Gujarat Titans                 648    1275        740
Kings XI Punjab               6227   14883       8954
Kochi Tuskers Kerala           337     884        680
Kolkata Knight Riders         7091   16799      10311
Lucknow Super Giants           643    1235        670
Mumbai Indians                8506   17608      10549
Pune Warriors                 1360    3103       1895
Punjab Kings                   905    2220       1368
Ra

Q8. Examine how index alignment behaves when
merging a bowler-wise wickets Series with a bowler-
wise runs-conceded Series. Compute economy rate
using aligned index arithmetic.

In [ ]:

# step1 loding dataset-------------------------------
# Step 2: Bowler-wise Wickets
# -------------------------------
wickets = df[df['isWicketDelivery'] == 1].groupby('bowler').size()

print("Wickets:\n", wickets.head())


# -------------------------------
# Step 3: Bowler-wise Runs Conceded
# -------------------------------
runs_conceded = df.groupby('bowler')['total_run'].sum()

print("\nRuns Conceded:\n", runs_conceded.head())


# -------------------------------
# Step 4: Balls Bowled (for economy)
# -------------------------------
df['valid_ball'] = np.where(df['extra_type'] == 'wides', 0, 1)

balls = df.groupby('bowler')['valid_ball'].sum()


# -------------------------------
# Step 5: Index Alignment Demonstration
# -------------------------------
# When performing arithmetic, pandas aligns by index automatically

combined = pd.DataFrame({
    'Runs': runs_conceded,
    'Wickets': wickets,
    'Balls': balls
})

print("\nCombined Data (Aligned by Index):\n", combined.head())


# -------------------------------
# Step 6: Handle Missing Values
# -------------------------------
combined = combined.fillna(0)


# -------------------------------
# Step 7: Compute Economy Rate
# -------------------------------
# Economy = Runs / Overs (Overs = balls / 6)

economy_rate = combined['Runs'] / (combined['Balls'] / 6)

combined['Economy'] = economy_rate

print("\nEconomy Rate:\n", combined[['Runs', 'Balls', 'Economy']].head())


# -------------------------------
# Step 8: Top Economical Bowlers
# -------------------------------
# Filter bowlers with minimum balls (e.g., 300 balls)
filtered = combined[combined['Balls'] > 300]

top_economy = filtered.sort_values(by='Economy').head(10)

print("\nTop Economical Bowlers:\n", top_economy[['Economy']])

Wickets:
 bowler
A Ashish Reddy    19
A Badoni           2
A Chandila        11
A Choudhary        5
A Flintoff         2
dtype: int64

Runs Conceded:
 bowler
A Ashish Reddy    400
A Badoni           12
A Chandila        245
A Choudhary       144
A Dananjaya        47
Name: total_run, dtype: int64

Combined Data (Aligned by Index):
                 Runs  Wickets  Balls
bowler                              
A Ashish Reddy   400     19.0    264
A Badoni          12      2.0     12
A Chandila       245     11.0    234
A Choudhary      144      5.0    102
A Dananjaya       47      NaN     24

Economy Rate:
                 Runs  Balls    Economy
bowler                                
A Ashish Reddy   400    264   9.090909
A Badoni          12     12   6.000000
A Chandila       245    234   6.282051
A Choudhary      144    102   8.470588
A Dananjaya       47     24  11.750000

Top Economical Bowlers:
                    Economy
bowler                    
Rashid Khan       6.551630
A Kumble  

Q9. Using hierarchical indexing on season and team,
examine total runs, wickets, and extras per season per
team. Use unstack() to build a season comparison
matrix.

In [ ]:


# -------------------------------
# Step 2: Create Season Column
# -------------------------------
# Skip if already present
df['season'] = df['ID'] // 1000000


# =====================================================
# 🔹 Step 3: Compute Metrics
# =====================================================

# Total Runs per team per season
runs = df.groupby(['season', 'BattingTeam'])['total_run'].sum()

# Total Wickets per team per season (bowling side)
wickets = df[df['isWicketDelivery'] == 1]\
            .groupby(['season', 'bowler']).size()

# Total Extras per team per season
extras = df.groupby(['season', 'BattingTeam'])['extras_run'].sum()


# =====================================================
# 🔹 Step 4: Combine into MultiIndex DataFrame
# =====================================================

combined = pd.DataFrame({
    'Runs': runs,
    'Extras': extras
})

# For wickets, align index properly (rename bowler → team level approx)
combined['Wickets'] = wickets

# Fill missing values
combined = combined.fillna(0)

print("Hierarchical Data:\n", combined.head())


# =====================================================
# 🔹 Step 5: Season Comparison Matrix using unstack()
# =====================================================

# Runs matrix
runs_matrix = combined['Runs'].unstack(level=0)

# Wickets matrix
wickets_matrix = combined['Wickets'].unstack(level=0)

# Extras matrix
extras_matrix = combined['Extras'].unstack(level=0)

print("\nRuns Comparison Matrix:\n", runs_matrix)
print("\nWickets Comparison Matrix:\n", wickets_matrix)
print("\nExtras Comparison Matrix:\n", extras_matrix)

Hierarchical Data:
                              Runs  Extras  Wickets
season BattingTeam                                
0      Chennai Super Kings  20899    1076      0.0
       Deccan Chargers      11463     578      0.0
       Delhi Daredevils     19734    1093      0.0
       Gujarat Lions         2450     132      0.0
       Kings XI Punjab      20861    1133      0.0

Runs Comparison Matrix:
 season                             0        1
BattingTeam                                  
Chennai Super Kings          20899.0  12494.0
Deccan Chargers              11463.0      NaN
Delhi Capitals                   NaN  10145.0
Delhi Daredevils             19734.0   4562.0
Gujarat Lions                 2450.0   2412.0
Gujarat Titans                   NaN   2663.0
Kings XI Punjab              20861.0   9203.0
Kochi Tuskers Kerala          1901.0      NaN
Kolkata Knight Riders        19481.0  14720.0
Lucknow Super Giants             NaN   2548.0
Mumbai Indians               21721.0  14942.0

Q10. Build a comprehensive IPL analytics report
covering null handling, hierarchical batting and bowling
analysis, ufunc performance metrics, and phase-wise
comparisons.

In [ ]:

# =====================================================
# 🔷 STEP 2: NULL HANDLING
# =====================================================
print("Missing Values:\n", df.isnull().sum())

# Fill dismissal-related nulls
df['player_out'] = df['player_out'].fillna('not out')
df['kind'] = df['kind'].fillna('not out')


# =====================================================
# 🔷 STEP 3: HIERARCHICAL BATTING ANALYSIS
# =====================================================
df['over_actual'] = df['overs'] + 1

batting_multi = df.groupby(['BattingTeam', 'over_actual'])['total_run'].sum()

print("\nBatting MultiIndex:\n", batting_multi.head())


# =====================================================
# 🔷 STEP 4: HIERARCHICAL BOWLING ANALYSIS
# =====================================================
# Valid balls
df['valid_ball'] = np.where(df['extra_type'] == 'wides', 0, 1)

runs_conceded = df.groupby('bowler')['total_run'].sum()
balls = df.groupby('bowler')['valid_ball'].sum()
wickets = df[df['isWicketDelivery'] == 1].groupby('bowler').size()

bowling_df = pd.DataFrame({
    'Runs': runs_conceded,
    'Balls': balls,
    'Wickets': wickets
}).fillna(0)

# Economy rate (ufunc alignment)
bowling_df['Economy'] = bowling_df['Runs'] / (bowling_df['Balls'] / 6)

print("\nBowling Analysis:\n", bowling_df.head())


# =====================================================
# 🔷 STEP 5: UFUNC PERFORMANCE METRICS (STRIKE RATE)
# =====================================================
runs = df.groupby('batter')['batsman_run'].sum()
balls_faced = df.groupby('batter')['valid_ball'].sum()

strike_rate = np.divide(runs, balls_faced) * 100

print("\nStrike Rate:\n", strike_rate.head())


# =====================================================
# 🔷 STEP 6: PLAYER IMPACT SCORE (UFUNC)
# =====================================================
fours = df[df['batsman_run'] == 4].groupby('batter').size()
sixes = df[df['batsman_run'] == 6].groupby('batter').size()

impact_df = pd.DataFrame({
    'runs': runs,
    'fours': fours,
    'sixes': sixes,
    'wickets': wickets
}).fillna(0)

# Normalize
impact_norm = (impact_df - impact_df.min()) / (impact_df.max() - impact_df.min())

# Weights
impact_df['Impact_Score'] = (
    np.multiply(impact_norm['runs'], 0.4) +
    np.multiply(impact_norm['fours'], 0.2) +
    np.multiply(impact_norm['sixes'], 0.2) +
    np.multiply(impact_norm['wickets'], 0.2)
)

print("\nTop Impact Players:\n", impact_df.sort_values(by='Impact_Score', ascending=False).head(10))


# =====================================================
# 🔷 STEP 7: PHASE-WISE ANALYSIS
# =====================================================
def phase(over):
    if over <= 5:
        return 'Powerplay'
    elif over <= 15:
        return 'Middle'
    else:
        return 'Death'

df['Phase'] = df['overs'].apply(phase)

phase_matrix = pd.pivot_table(
    df,
    values='total_run',
    index='BattingTeam',
    columns='Phase',
    aggfunc='sum'
)

print("\nPhase-wise Performance:\n", phase_matrix)


# =====================================================
# 🔷 STEP 8: EXTRAS ANALYSIS (BOOLEAN INDEXING)
# =====================================================
extras_df = df[df['extras_run'] > 0]

extras_by_team = extras_df.groupby('BattingTeam')['extras_run'].sum()
extras_by_phase = extras_df.groupby('Phase')['extras_run'].sum()

print("\nExtras by Team:\n", extras_by_team.sort_values(ascending=False).head())
print("\nExtras by Phase:\n", extras_by_phase)


# =====================================================
# 🔷 STEP 9: SEASON-WISE ANALYSIS (MULTIINDEX)
# =====================================================
df['season'] = df['ID'] // 1000000

season_data = df.groupby(['season', 'BattingTeam']).agg({
    'total_run': 'sum',
    'extras_run': 'sum'
})

season_data['wickets'] = df[df['isWicketDelivery'] == 1]\
                           .groupby(['season', 'BattingTeam']).size()

season_data = season_data.fillna(0)

# Unstack for comparison
season_matrix = season_data['total_run'].unstack(level=0)

print("\nSeason-wise Runs Matrix:\n", season_matrix)

Missing Values:
 ID                        0
innings                   0
overs                     0
ballnumber                0
batter                    0
bowler                    0
non-striker               0
extra_type           213905
batsman_run               0
extras_run                0
total_run                 0
non_boundary              0
isWicketDelivery          0
player_out           214803
kind                 214803
fielders_involved    217966
BattingTeam               0
valid_ball                0
over_actual               0
season                    0
BowlingTeam               0
Phase                     0
dtype: int64

Batting MultiIndex:
 BattingTeam          over_actual
Chennai Super Kings  1              1090
                     2              1353
                     3              1587
                     4              1732
                     5              1784
Name: total_run, dtype: int64

Bowling Analysis:
                 Runs  Balls  Wickets    Econ